<a href="https://colab.research.google.com/github/xwang335/Campbell-A/blob/main/nn1_final_version.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Neural Network NN1 v2 — Gu, Kelly & Xiu (2020)

**Aligned with paper specification (Appendix B.3, Table A.5):**

| Parameter | Paper | This code |
|-----------|-------|-----------|
| Optimizer | Adam | Adam |
| LR grid | {0.01, 0.1} | {0.01, 0.1} |
| Regularization | L1 penalty, λ ∈ {1e-5, 1e-3} | L1 (manual) |
| Batch Norm | Yes (per hidden layer) | Yes |
| Dropout | No | No |
| Batch size | 10,000 | 10,000 |
| Max epochs | 100 | 100 |
| Early stopping patience | 5 | 5 |
| Ensemble seeds | 10 | 10 |
| Gradient clipping | max_norm=1.0 | max_norm=1.0 |
| LR schedule | None (plain Adam) | None |

**Additional (not in paper):**
- Target winsorization: 0.5%/99.5% cross-sectional clip per month
- AMP (mixed precision) for GPU acceleration
- Checkpoint resume across sessions

HP combos: 2 L1 × 2 LR = 4 combos × 10 seeds = **40 models/year**

In [1]:
import os, gc, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print('GPU:', torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    DEVICE = torch.device('cpu')
    print('Using CPU')

USE_AMP = DEVICE.type == 'cuda'
print('Device:', DEVICE)
print('AMP enabled:', USE_AMP)
print('Torch:', torch.__version__)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: NVIDIA A100-SXM4-80GB
Device: cuda
AMP enabled: True
Torch: 2.10.0+cu128


In [2]:
# -------- paths --------
PARQUET_PATH = '/content/drive/MyDrive/Campbell A data/preprocess_data.parquet'
OUTPUT_DIR   = '/content/drive/MyDrive/backtest'

NN_ARCH = 'NN1'  # options: 'NN1', 'NN2', 'NN3', 'NN4', 'NN5'

OUTPUT_PATH    = f'{OUTPUT_DIR}/{NN_ARCH.lower()}_v2_results.parquet'
YEAR_INFO_PATH = f'{OUTPUT_DIR}/{NN_ARCH.lower()}_v2_year_info.csv'

os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_parquet(PARQUET_PATH)
df['DATE'] = pd.to_datetime(df['DATE'])

print('shape:', df.shape)
print('date range:', df['DATE'].min().date(), 'to', df['DATE'].max().date())
print('permno count:', df['permno'].nunique())
print(f'\nRunning architecture: {NN_ARCH}')
df.head(3)

shape: (3712808, 109)
date range: 1957-04-30 to 2016-12-30
permno count: 29825

Running architecture: NN1


,permno,DATE,mvel1,beta,betasq,chmom,dolvol,idiovol,indmom,mom1m,...,exret,exret_lead1,tbl,d/p,e/p,b/m,tms,dfy,ntis,svar
0,10000,1986-02-28,-0.381006,0.0,0.0,0.0,0.000000,0.0,0.310144,0.000485,...,-0.262443,0.359385,0.0707,0.037492,0.068845,0.583517,0.0251,0.0139,-0.019172,0.001920
1,10000,1986-03-31,-0.505829,0.0,0.0,0.0,0.000000,0.0,0.358323,-0.952720,...,0.359385,-0.103792,0.0706,0.035167,0.064120,0.536377,0.0135,0.0144,-0.017914,0.001089
2,10000,1986-04-30,-0.410132,0.0,0.0,0.0,-0.539206,0.0,0.281704,0.932882,...,-0.103792,-0.227556,0.0656,0.033571,0.060779,0.519628,0.0110,0.0150,-0.016420,0.001374


In [3]:
# -------- column sets (identical to ENet) --------
NON_CHAR_COLS = {
    'permno', 'DATE', 'ret', 'rf', 'exret', 'exret_lead1', 'sic2',
    'tbl', 'b/m', 'd/p', 'e/p', 'ntis', 'tms', 'dfy', 'svar'
}
MACRO_COLS = ['tbl', 'b/m', 'd/p', 'e/p', 'ntis', 'tms', 'dfy', 'svar']
CHAR_COLS_94 = sorted([c for c in df.columns if c not in NON_CHAR_COLS])

sic2_dummies = pd.get_dummies(df['sic2'].astype(str), prefix='sic2', drop_first=False)
SIC2_DUMMY_COLS = sorted(sic2_dummies.columns.tolist())
df = pd.concat([df, sic2_dummies], axis=1)
del sic2_dummies
gc.collect()

print('94 stock characteristics:', len(CHAR_COLS_94))
print('8 macro variables:', MACRO_COLS)
print('SIC2 dummy cols:', len(SIC2_DUMMY_COLS))
print('total features:', len(CHAR_COLS_94) * 9 + len(SIC2_DUMMY_COLS))

94 stock characteristics: 94
8 macro variables: ['tbl', 'b/m', 'd/p', 'e/p', 'ntis', 'tms', 'dfy', 'svar']
SIC2 dummy cols: 74
total features: 920


In [4]:
# -------- model config (aligned with GKX 2020, Table A.5) --------
TARGET = 'exret_lead1'
VALIDATION_END = pd.Timestamp('1986-12-31')
TEST_END = pd.Timestamp('2016-12-31')
VALIDATION_YEARS = 12

NN_ARCHITECTURES = {
    'NN1': [32],
    'NN2': [32, 16],
    'NN3': [32, 16, 8],
    'NN4': [32, 16, 8, 4],
    'NN5': [32, 16, 8, 4, 2],
}

HIDDEN_LAYERS = NN_ARCHITECTURES[NN_ARCH]

# HP grid — matches paper Table A.5
L1_GRID = [1e-5, 1e-3]             # L1 penalty candidates
LR_GRID = [0.01, 0.1]              # learning rate candidates
BATCH_SIZE = 10_000                 # paper: 10,000
MAX_EPOCHS = 100                    # paper: 100
PATIENCE = 5                        # paper: 5 (Algorithm 6)
N_ENSEMBLE = 10                     # paper: 10 seeds
GRAD_CLIP = 1.0

REQUIRED = CHAR_COLS_94 + MACRO_COLS + [TARGET, 'mvel1']
df_clean = df.dropna(subset=REQUIRED).copy()
df_clean = df_clean.sort_values(['DATE', 'permno']).reset_index(drop=True)

# -------- Cross-sectional winsorization of target (per month) --------
# n_before = len(df_clean)
# lo_hi = df_clean.groupby('DATE')[TARGET].transform(
#     lambda s: pd.Series(
#         s.clip(lower=s.quantile(0.005), upper=s.quantile(0.995)),
#         index=s.index,
#     )
# )
# n_clipped = (df_clean[TARGET] != lo_hi).sum()
# df_clean[TARGET] = lo_hi
# del lo_hi
# print(f'Target winsorized: {n_clipped:,} values clipped ({n_clipped/n_before*100:.2f}%)')

test_years = sorted(
    df_clean.loc[
        (df_clean['DATE'] > VALIDATION_END) & (df_clean['DATE'] <= TEST_END),
        'DATE'
    ].dt.year.unique()
)

print('clean rows:', f'{len(df_clean):,}')
print('test years:', test_years[0], 'to', test_years[-1], f'({len(test_years)})')
print()
print(f'Architecture: {NN_ARCH} = {HIDDEN_LAYERS}')
print(f'L1 grid: {L1_GRID}')
print(f'LR grid: {LR_GRID}')
print(f'Batch size: {BATCH_SIZE}, Max epochs: {MAX_EPOCHS}, Patience: {PATIENCE}')
print(f'Gradient clip: {GRAD_CLIP}')
print(f'HP combos: {len(L1_GRID)*len(LR_GRID)} x {N_ENSEMBLE} seeds = {len(L1_GRID)*len(LR_GRID)*N_ENSEMBLE} models/year')

clean rows: 3,712,808
test years: 1987 to 2016 (30)

Architecture: NN1 = [32]
L1 grid: [1e-05, 0.001]
LR grid: [0.01, 0.1]
Batch size: 10000, Max epochs: 100, Patience: 5
Gradient clip: 1.0
HP combos: 4 x 10 seeds = 40 models/year


In [5]:
# -------- Precompute features + free DataFrame RAM --------

print('Precomputing 920-dim feature matrix for all rows...')
t0 = time.time()

chars_all = df_clean[CHAR_COLS_94].to_numpy(dtype=np.float32)
macro_with_const_all = np.column_stack([
    np.ones(len(df_clean), dtype=np.float32),
    df_clean[MACRO_COLS].to_numpy(dtype=np.float32)
])
interactions_all = (chars_all[:, :, None] * macro_with_const_all[:, None, :]).reshape(len(df_clean), -1)
sic2_all = df_clean[SIC2_DUMMY_COLS].to_numpy(dtype=np.float32)
X_all = np.hstack([interactions_all, sic2_all])
y_all = df_clean[TARGET].to_numpy(dtype=np.float64)

# Store year + meta columns for output
year_col    = df_clean['DATE'].dt.year.to_numpy()
meta_date   = df_clean['DATE'].to_numpy()
meta_permno = df_clean['permno'].to_numpy()
meta_mvel1  = df_clean['mvel1'].to_numpy(dtype=np.float64)

del chars_all, macro_with_const_all, interactions_all, sic2_all
del df, df_clean
gc.collect()

print(f'X_all shape: {X_all.shape}, dtype: {X_all.dtype}')
print(f'Precompute time: {time.time()-t0:.1f}s')
print(f'Memory: {X_all.nbytes / 1e9:.2f} GB')
print('df and df_clean deleted')

Precomputing 920-dim feature matrix for all rows...
X_all shape: (3712808, 920), dtype: float32
Precompute time: 11.9s
Memory: 13.66 GB
df and df_clean deleted


In [6]:
# -------- helpers --------

def standardize_inplace(X, mean=None, std=None):
    """Standardize in-place. Returns mean, std."""
    if mean is None:
        mean = X.mean(axis=0)
    if std is None:
        std = X.std(axis=0)
    std = np.where(std < 1e-8, 1.0, std)
    X -= mean
    X /= std
    return mean, std

def oos_r2(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    denom = np.sum(y_true ** 2)
    if denom == 0:
        return np.nan
    return 1.0 - np.sum((y_true - y_pred) ** 2) / denom

print('helpers ready')

helpers ready


In [7]:
# -------- NN Model Definition (paper: Linear → BN → ReLU, no Dropout) --------

class AssetPricingNN(nn.Module):
    def __init__(self, input_dim, hidden_layers):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h_dim in hidden_layers:
            layers.append(nn.Linear(prev_dim, h_dim))
            layers.append(nn.BatchNorm1d(h_dim))
            layers.append(nn.ReLU())
            prev_dim = h_dim
        layers.append(nn.Linear(prev_dim, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x).squeeze(-1)

test_model = AssetPricingNN(920, HIDDEN_LAYERS)
print(f'{NN_ARCH} architecture:')
print(test_model)
print(f'\nTotal parameters: {sum(p.numel() for p in test_model.parameters()):,}')
del test_model

NN1 architecture:
AssetPricingNN(
  (network): Sequential(
    (0): Linear(in_features=920, out_features=32, bias=True)
    (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Linear(in_features=32, out_features=1, bias=True)
  )
)

Total parameters: 29,569


In [8]:
# -------- Training function (paper: Adam + L1 + early stopping + BN) --------

def train_single_nn(X_train, y_train, X_val, y_val,
                    hidden_layers, l1_lambda, lr,
                    batch_size, max_epochs, patience, seed, device,
                    grad_clip=1.0, use_amp=False):
    """
    Train one NN with Adam + manual L1 penalty + BN + early stopping + AMP.
    Matches GKX 2020 Appendix B.3 / Algorithm 5-6.
    """
    torch.manual_seed(seed)
    np.random.seed(seed)
    if device.type == 'cuda':
        torch.cuda.manual_seed(seed)

    input_dim = X_train.shape[1]
    n_train = X_train.shape[0]
    model = AssetPricingNN(input_dim, hidden_layers).to(device)

    # Plain Adam (no weight decay — L1 is applied manually)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    scaler = GradScaler(enabled=use_amp)

    best_val_loss = float('inf')
    best_state = None
    best_epoch = 0
    epochs_no_improve = 0

    for epoch in range(max_epochs):
        model.train()
        perm = torch.randperm(n_train, device=device)

        for i in range(0, n_train, batch_size):
            idx = perm[i:i+batch_size]
            X_batch = X_train[idx]
            y_batch = y_train[idx]

            optimizer.zero_grad(set_to_none=True)

            with autocast(enabled=use_amp):
                y_pred = model(X_batch)
                mse_loss = nn.functional.mse_loss(y_pred, y_batch)

                # Manual L1 penalty on all weight parameters (not biases, not BN params)
                l1_loss = torch.tensor(0.0, device=device)
                for name, param in model.named_parameters():
                    if 'weight' in name and 'bn' not in name.lower() and 'norm' not in name.lower():
                        l1_loss = l1_loss + param.abs().sum()

                loss = mse_loss + l1_lambda * l1_loss

            scaler.scale(loss).backward()

            # Gradient clipping
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

            scaler.step(optimizer)
            scaler.update()

        # Validation (MSE only, no L1)
        model.eval()
        with torch.no_grad():
            with autocast(enabled=use_amp):
                val_pred = model(X_val)
            val_loss = nn.functional.mse_loss(val_pred.float(), y_val).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            best_epoch = epoch
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, {'best_epoch': best_epoch, 'best_val_loss': best_val_loss,
                   'total_epochs': epoch + 1}


def predict_ensemble(models, X, device, use_amp=False):
    preds = []
    for model in models:
        model.eval()
        with torch.no_grad():
            with autocast(enabled=use_amp):
                pred = model(X).float().cpu().numpy()
            preds.append(pred)
    return np.mean(preds, axis=0)

print('training functions ready (paper: Adam + L1 + BN + early stopping)')

training functions ready (paper: Adam + L1 + BN + early stopping)


## Main Estimation Loop (v2)

**Full HP search:** 4 combos (2 L1 × 2 LR) × 10 seeds = 40 models/year.  
Pick best HP by ensemble validation MSE → use that ensemble for test prediction.

In [9]:
# ===== CHECKPOINT: setup directory + detect completed years =====
# v2c: aligned with paper (L1 + BN + Adam + patience=5)
CKPT_DIR = f"{OUTPUT_DIR}/{NN_ARCH.lower()}_v2c_ckpt"
os.makedirs(CKPT_DIR, exist_ok=True)

done_years = set()
for fname in os.listdir(CKPT_DIR):
    if fname.endswith('_pred.parquet'):
        try:
            done_years.add(int(fname.split('_')[0]))
        except ValueError:
            pass

if done_years:
    print(f"Checkpoint: {len(done_years)} years already done: {sorted(done_years)}")
else:
    print("Checkpoint: starting fresh")

year_info = []

for year in test_years:
    # ===== CHECKPOINT: skip completed years =====
    if year in done_years:
        ckpt_info_path = f"{CKPT_DIR}/{year}_info.parquet"
        if os.path.exists(ckpt_info_path):
            info_row = pd.read_parquet(ckpt_info_path).iloc[0].to_dict()
            year_info.append(info_row)
        print(f'Year {year} | loaded from checkpoint')
        continue

    t0 = time.time()

    # ---- Sample split ----
    train_end_year = year - VALIDATION_YEARS - 1
    val_start_year = year - VALIDATION_YEARS
    val_end_year   = year - 1

    mask_train = year_col <= train_end_year
    mask_val   = (year_col >= val_start_year) & (year_col <= val_end_year)
    mask_test  = year_col == year

    X_train_np = X_all[mask_train].copy()
    X_val_np   = X_all[mask_val].copy()
    X_test_np  = X_all[mask_test].copy()
    y_train_np = y_all[mask_train]
    y_val_np   = y_all[mask_val]
    y_test_np  = y_all[mask_test]

    # Standardize (fit on train only)
    # x_mu, x_sd = standardize_inplace(X_train_np)
    # standardize_inplace(X_val_np, x_mu, x_sd)
    # standardize_inplace(X_test_np, x_mu, x_sd)

    # Demean target
    y_mu = float(y_train_np.mean())
    y_tr_c = (y_train_np - y_mu).astype(np.float32)
    y_va_c = (y_val_np   - y_mu).astype(np.float32)

    # Move to GPU
    X_tr = torch.tensor(X_train_np, dtype=torch.float32, device=DEVICE)
    del X_train_np; gc.collect()
    X_va = torch.tensor(X_val_np,   dtype=torch.float32, device=DEVICE)
    del X_val_np; gc.collect()
    X_te = torch.tensor(X_test_np,  dtype=torch.float32, device=DEVICE)
    del X_test_np; gc.collect()
    y_tr = torch.tensor(y_tr_c,    dtype=torch.float32, device=DEVICE)
    y_va = torch.tensor(y_va_c,    dtype=torch.float32, device=DEVICE)
    del y_tr_c, y_va_c; gc.collect()

    # ======================================================
    # Full HP search: all 4 combos × 10 seeds
    # ======================================================
    best_hp_mse = float('inf')
    best_l1 = L1_GRID[0]
    best_lr = LR_GRID[0]
    best_models = None
    best_epochs_list = None

    for l1_lambda in L1_GRID:
        for lr in LR_GRID:
            hp_models = []
            hp_epochs = []
            for seed_idx in range(N_ENSEMBLE):
                seed = seed_idx * 1000 + year
                model, hist = train_single_nn(
                    X_tr, y_tr, X_va, y_va,
                    hidden_layers=HIDDEN_LAYERS,
                    l1_lambda=l1_lambda, lr=lr,
                    batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS,
                    patience=PATIENCE, seed=seed, device=DEVICE,
                    grad_clip=GRAD_CLIP, use_amp=USE_AMP
                )
                hp_models.append(model)
                hp_epochs.append(hist['best_epoch'])

            # Ensemble validation MSE
            val_pred = predict_ensemble(hp_models, X_va, DEVICE, USE_AMP)
            val_pred_t = torch.tensor(val_pred, dtype=torch.float32, device=DEVICE)
            hp_mse = nn.functional.mse_loss(val_pred_t, y_va).item()

            if hp_mse < best_hp_mse:
                best_hp_mse = hp_mse
                best_l1 = l1_lambda
                best_lr = lr
                if best_models is not None:
                    del best_models
                best_models = hp_models
                best_epochs_list = hp_epochs
            else:
                del hp_models

            del val_pred, val_pred_t
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    best_avg_epochs = float(np.mean(best_epochs_list))

    # ---- Predict on test set ----
    y_pred_ensemble = predict_ensemble(best_models, X_te, DEVICE, USE_AMP)
    y_pred = y_pred_ensemble.astype(np.float64) + y_mu

    # ---- Store results ----
    res = pd.DataFrame({
        'DATE':   meta_date[mask_test],
        'permno': meta_permno[mask_test],
        'mvel1':  meta_mvel1[mask_test],
        'y_true': y_test_np,
        'y_pred': y_pred,
    })

    elapsed = time.time() - t0

    info = {
        'year': year,
        'n_train': int(mask_train.sum()),
        'n_val': int(mask_val.sum()),
        'n_test': int(mask_test.sum()),
        'best_l1': best_l1,
        'best_lr': best_lr,
        'best_val_mse': best_hp_mse,
        'avg_best_epoch': best_avg_epochs,
        'sec': elapsed,
    }
    year_info.append(info)

    # ===== CHECKPOINT: save to Drive =====
    res.to_parquet(f"{CKPT_DIR}/{year}_pred.parquet", index=False)
    pd.DataFrame([info]).to_parquet(f"{CKPT_DIR}/{year}_info.parquet", index=False)

    print(
        f'Year {year} | train {int(mask_train.sum()):>8,} | val {int(mask_val.sum()):>8,} | '
        f'test {int(mask_test.sum()):>7,} | '
        f'l1={best_l1:.0e} lr={best_lr} '
        f'avg_epoch={best_avg_epochs:.0f} '
        f'val_mse={best_hp_mse:.6f} | {elapsed:.1f}s [saved]'
    )

    del X_tr, X_va, X_te, y_tr, y_va, best_models, res, y_pred, y_pred_ensemble
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f'\n{NN_ARCH} v2 estimation complete.')

Checkpoint: starting fresh
Year 1987 | train  472,278 | val  764,497 | test  82,404 | l1=1e-03 lr=0.1 avg_epoch=4 val_mse=0.024778 | 74.3s [saved]
Year 1988 | train  530,435 | val  788,744 | test  83,415 | l1=1e-05 lr=0.01 avg_epoch=3 val_mse=0.025673 | 79.1s [saved]
Year 1989 | train  588,534 | val  814,060 | test  81,216 | l1=1e-05 lr=0.01 avg_epoch=2 val_mse=0.026219 | 73.9s [saved]
Year 1990 | train  647,363 | val  836,447 | test  80,207 | l1=1e-05 lr=0.01 avg_epoch=4 val_mse=0.027091 | 78.5s [saved]
Year 1991 | train  704,916 | val  859,101 | test  79,274 | l1=1e-05 lr=0.01 avg_epoch=3 val_mse=0.028452 | 90.2s [saved]
Year 1992 | train  761,970 | val  881,321 | test  80,972 | l1=1e-05 lr=0.01 avg_epoch=2 val_mse=0.032443 | 108.3s [saved]
Year 1993 | train  819,497 | val  904,766 | test  86,150 | l1=1e-05 lr=0.01 avg_epoch=2 val_mse=0.032913 | 96.3s [saved]
Year 1994 | train  880,933 | val  929,480 | test  95,088 | l1=1e-05 lr=0.01 avg_epoch=4 val_mse=0.033587 | 100.4s [saved]
Year

In [10]:
# -------- Aggregate results & OOS R^2 --------
del X_all, y_all, year_col, meta_date, meta_permno, meta_mvel1
gc.collect()

all_preds = []
for year in test_years:
    ckpt_path = f"{CKPT_DIR}/{year}_pred.parquet"
    all_preds.append(pd.read_parquet(ckpt_path))

results = pd.concat(all_preds, ignore_index=True)
del all_preds; gc.collect()

results['DATE'] = pd.to_datetime(results['DATE'])
results = results.sort_values(['DATE', 'permno']).reset_index(drop=True)

year_info_df = pd.DataFrame(year_info)

print(f'=== {NN_ARCH} v2 Results ===')
print('total predictions:', f'{len(results):,}')
print('test period:', results['DATE'].min().date(), 'to', results['DATE'].max().date())

r2_all = oos_r2(results['y_true'], results['y_pred'])

top1000 = (
    results.sort_values(['DATE', 'mvel1'], ascending=[True, False])
           .groupby('DATE', sort=False)
           .head(1000)
)
bot1000 = (
    results.sort_values(['DATE', 'mvel1'], ascending=[True, True])
           .groupby('DATE', sort=False)
           .head(1000)
)

r2_top = oos_r2(top1000['y_true'], top1000['y_pred'])
r2_bot = oos_r2(bot1000['y_true'], bot1000['y_pred'])

paper_ref = {
    'NN1': 0.11, 'NN2': 0.12, 'NN3': 0.20, 'NN4': 0.17, 'NN5': 0.12,
}
paper_val = paper_ref.get(NN_ARCH, '?')

print('=' * 72)
print(f'{"Subsample":<35} {"OOS R^2":>12} {f"Paper {NN_ARCH}":>18}')
print('-' * 72)
print(f'{"All stocks (panel)":<35} {r2_all*100:>+11.4f}% {f"~ +{paper_val}%":>18}')
print(f'{"Top 1,000 (largest mvel1)":<35} {r2_top*100:>+11.4f}%')
print(f'{"Bottom 1,000 (smallest mvel1)":<35} {r2_bot*100:>+11.4f}%')
print('=' * 72)

print()
print('HP selection summary:')
print(f'  l1_lambda distribution:')
for l1_val in sorted(L1_GRID):
    cnt = (year_info_df["best_l1"] == l1_val).sum()
    if cnt > 0:
        print(f'    {cnt} years chose l1={l1_val:.0e}')
print(f'  lr distribution:')
for lr_val in sorted(LR_GRID):
    cnt = (year_info_df["best_lr"] == lr_val).sum()
    if cnt > 0:
        print(f'    {cnt} years chose lr={lr_val}')
print(f'  avg_best_epoch median: {year_info_df["avg_best_epoch"].median():.0f}')

=== NN1 v2 Results ===
total predictions: 2,476,033
test period: 1987-01-30 to 2016-12-30
Subsample                                OOS R^2          Paper NN1
------------------------------------------------------------------------
All stocks (panel)                      +0.0794%           ~ +0.11%
Top 1,000 (largest mvel1)               +0.0764%
Bottom 1,000 (smallest mvel1)           +0.1272%

HP selection summary:
  l1_lambda distribution:
    29 years chose l1=1e-05
    1 years chose l1=1e-03
  lr distribution:
    28 years chose lr=0.01
    2 years chose lr=0.1
  avg_best_epoch median: 4


## Year-by-year diagnostics

In [11]:
print(f'Year-by-year results ({NN_ARCH} v2):')
print(year_info_df[['year', 'best_l1', 'best_lr', 'best_val_mse',
                     'avg_best_epoch', 'sec']].to_string(index=False))

yearly_r2 = []
for year in test_years:
    mask = results['DATE'].dt.year == year
    r2_y = oos_r2(results.loc[mask, 'y_true'], results.loc[mask, 'y_pred'])
    yearly_r2.append({'year': year, 'oos_r2': r2_y})

yearly_r2_df = pd.DataFrame(yearly_r2)
year_info_df = year_info_df.merge(yearly_r2_df, on='year')

print(f'\nYearly OOS R^2:')
print(yearly_r2_df.to_string(index=False))

Year-by-year results (NN1 v2):
 year  best_l1  best_lr  best_val_mse  avg_best_epoch        sec
 1987  0.00100     0.10      0.024778             4.5  74.250942
 1988  0.00001     0.01      0.025673             2.8  79.136800
 1989  0.00001     0.01      0.026219             2.4  73.877902
 1990  0.00001     0.01      0.027091             3.5  78.465600
 1991  0.00001     0.01      0.028452             3.1  90.211990
 1992  0.00001     0.01      0.032443             2.2 108.267249
 1993  0.00001     0.01      0.032913             1.7  96.299667
 1994  0.00001     0.01      0.033587             3.5 100.409125
 1995  0.00001     0.01      0.032568             3.2 114.590600
 1996  0.00001     0.01      0.032482             3.9 121.529946
 1997  0.00001     0.01      0.032745             3.0 119.623089
 1998  0.00001     0.01      0.032752             1.7 118.846074
 1999  0.00001     0.01      0.034843             4.5 142.980324
 2000  0.00001     0.01      0.036644             0.4 133.7

In [12]:
# -------- Save outputs --------
cols_to_save = ['permno', 'DATE', 'y_true', 'y_pred', 'mvel1']
results[cols_to_save].to_parquet(OUTPUT_PATH, index=False)
year_info_df.to_csv(YEAR_INFO_PATH, index=False)

print(f'saved prediction parquet: {OUTPUT_PATH}')
print(f'saved yearly diagnostics: {YEAR_INFO_PATH}')
results.head()

saved prediction parquet: /content/drive/MyDrive/backtest/nn1_v2_results.parquet
saved yearly diagnostics: /content/drive/MyDrive/backtest/nn1_v2_year_info.csv


,DATE,permno,mvel1,y_true,y_pred
0,1987-01-30,10000,-0.921324,-0.004300,0.004578
1,1987-01-30,10001,-0.671628,-0.078374,0.004578
2,1987-01-30,10002,-0.413730,-0.018125,0.004578
3,1987-01-30,10003,0.030073,0.007194,0.004578
4,1987-01-30,10005,-0.994228,0.095700,0.004578
